# 04_feature_engineering

One notebook, one purpose: turn the cleaned listings into the exact feature set
`03_insights.ipynb` called for, for a **price-prediction model** — for both datasets. Every
column decision below (`KEEP` / `DROP` / `ENGINEER`) is taken directly from that notebook's
Part A and Part B summary tables.

**Scope, kept deliberately narrow:** no train/test split, no one-hot/target encoding, no
scaling, no model fitting here — those belong in the modeling notebook that consumes these
two files, once it's clear which model (linear vs. tree-based) goes first. What *does* belong
here: actually building the specific engineered features and interaction terms the insights
notebook called for by name, not just noting them as "modeling cautions" to revisit later.

**Two interaction terms get implemented here that didn't exist in the previous draft** — the
insights notebook flagged both `one_owner`/age-mileage and `transmission`/`Make` as confounds
needing "an interaction term or a tree-based model," and `manufacturer`'s ENGINEER call
specifically named the depreciation-rate interaction. All three are built below, not just
left as a caution in a markdown cell.

Output:
- `../data/processed/cars_features.csv`
- `../data/processed/used_cars_features.csv`


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
CURRENT_YEAR = 2026


## Part A — `cars` dataset (US)

In [2]:
df_a = pd.read_csv("../data/processed/cars_clean.csv", low_memory=False)
print("Shape:", df_a.shape)


Shape: (752900, 24)


### Implementing the Part A summary table

| Feature | Call (from `03_insights.ipynb`) | What's done here |
|---|---|---|
| `accidents_or_damage` | KEEP | used as-is |
| `one_owner` | KEEP, caution | used as-is, **plus** an explicit interaction term with age/mileage (below) — the caution was "confounded, needs an interaction term," so this builds it rather than leaving it as a note |
| `personal_use_only` | already dropped in cleaning | nothing to do |
| `price_drop` / `has_price_drop` | **DROP** | not carried into the feature set at all |
| `manufacturer` | **KEEP, engineer interaction** | frequency-encoded, **plus** a `manufacturer × age` interaction term (below) — same reasoning as `one_owner`: the insight named this interaction specifically, so it gets built, not just flagged |
| `model` | **ENGINEER** | frequency-encoded (12,186 raw values) |
| `city_mpg` / `highway_mpg` | KEEP both | used as-is (0.47 correlated, not redundant) |
| `seller_rating` / `driver_rating` | **DROP** | not carried into the feature set — but their `_was_missing` flags are kept, per Insight A6's explicit note that those track listing completeness, a separate signal from price |
| `driver_reviews_num` | KEEP (test) | used as-is (-0.27 correlation, the strongest of the three rating fields) |
| `price` | ENGINEER | `price_log` added as the modeling target |

`engine` and `transmission` weren't in the insights notebook's summary table at all — they
came up while actually building this notebook. `engine` is free text (e.g. `"1.5L I-4
i-VTEC..., engine with 90HP"`) with real displacement/cylinder/turbo/hybrid signal sitting
in it unparsed; `transmission` is similarly free text (1,314 unique values) rather than a
clean category.

In [3]:
df_a['age'] = CURRENT_YEAR - df_a['year']
df_a['price_log'] = np.log1p(df_a['price'])

def simplify_transmission(t):
    t = str(t).lower()
    if 'cvt' in t or 'variable' in t:
        return 'CVT'
    if 'manual' in t or t == 'm/t':
        return 'Manual'
    if 'automatic' in t or 'a/t' in t or 'auto' in t:
        return 'Automatic'
    return 'Other/Unknown'

df_a['transmission_simple'] = df_a['transmission'].apply(simplify_transmission)

speed_extracted = df_a['transmission'].str.extract(r'(\d+)-?[Ss]peed')[0]
df_a['transmission_num_speeds'] = pd.to_numeric(speed_extracted, errors='coerce')

print(df_a['transmission_simple'].value_counts())
print()
print(df_a['transmission_num_speeds'].describe())


transmission_simple
Automatic        598313
CVT              116222
Manual            22934
Other/Unknown     15431
Name: count, dtype: int64

count    511965.000000
mean          7.075204
std           1.805175
min           1.000000
25%           6.000000
50%           7.000000
75%           8.000000
max          10.000000
Name: transmission_num_speeds, dtype: float64


### `engine` → parsed spec features (new -- not in `03_insights.ipynb`'s table)

Displacement, cylinder count, turbo, and hybrid flags pulled out of the free text via regex.
`pd.to_numeric(errors='coerce')` rather than `.astype(float)`, since values like `"Unknown"`
or `"0 SELECT"` won't match any pattern and should become `NaN`, not crash the cell.

In [4]:
disp_extracted = df_a['engine'].str.extract(r'(\d+\.?\d*)\s*[Ll]\b')[0]
df_a['engine_displacement_l'] = pd.to_numeric(disp_extracted, errors='coerce')

cyl_extracted = df_a['engine'].str.extract(r'[IVH]-?(\d+)|(\d+)-?[Cc]yl').bfill(axis=1).iloc[:, 0]
df_a['engine_cylinders'] = pd.to_numeric(cyl_extracted, errors='coerce')

df_a['engine_is_turbo'] = df_a['engine'].str.contains('turbo', case=False, na=False).astype(int)
df_a['engine_is_hybrid'] = (
    df_a['engine'].str.contains('hybrid', case=False, na=False) |
    df_a['fuel_type'].str.contains('Hybrid', case=False, na=False)
).astype(int)

df_a[['engine_displacement_l', 'engine_cylinders', 'engine_is_turbo', 'engine_is_hybrid']].describe()


,engine_displacement_l,engine_cylinders,engine_is_turbo,engine_is_hybrid
count,708986.000000,706305.000000,752900.000000,752900.000000
mean,3.102593,5.294216,0.307564,0.038852
std,1.426665,1.478371,0.461485,0.193243
min,0.000000,0.000000,0.000000,0.000000
25%,2.000000,4.000000,0.000000,0.000000
50%,2.500000,5.000000,0.000000,0.000000
75%,3.600000,6.000000,1.000000,0.000000
max,350.000000,22.000000,1.000000,1.000000


### `model` and `manufacturer` → frequency encoding (Insight A7's ENGINEER call)

12,186 unique raw model strings is too high-cardinality for one-hot, and target encoding needs
a train/test split to avoid leakage that doesn't belong in a feature-engineering notebook.
Frequency encoding — how common a model or manufacturer is in the dataset — captures
"well-known vs. rare" without touching the target, so it's safe to compute here.

`manufacturer` only has 30 categories, small enough to one-hot directly later — but it's
frequency-encoded here too, specifically so it can feed into the interaction term below.

In [5]:
model_freq = df_a['model'].value_counts(normalize=True)
df_a['model_freq'] = df_a['model'].map(model_freq)

manufacturer_freq = df_a['manufacturer'].value_counts(normalize=True)
df_a['manufacturer_freq'] = df_a['manufacturer'].map(manufacturer_freq)

print(df_a[['model', 'model_freq']].drop_duplicates().sort_values('model_freq', ascending=False).head(5))


                      model  model_freq
248122            Fusion SE    0.004213
413545          Sportage LX    0.003781
692397           Corolla LE    0.003764
505915  GLC 300 Base 4MATIC    0.003542
582680            Sentra SV    0.003501


### Interaction terms (Insight A2 and A4's ENGINEER/caution calls, actually built)

- **`manufacturer_age_interaction`** — Insight A4: depreciation rate varies sharply by brand
  (-1.4%/yr Chevrolet vs. -10.9%/yr Kia). A tree-based model finds this automatically from
  `manufacturer_freq` and `age` as separate columns, but a linear model needs it spelled out.
- **`one_owner_age_interaction`** / **`one_owner_mileage_interaction`** — Insight A2: about
  half the raw 32.9% one-owner price premium is really an age/mileage confound (one-owner
  cars are ~3 years newer with roughly half the mileage). These interaction terms let a
  linear model separate "genuinely one-owner effect" from "just a newer, lower-mileage car
  that happens to be one-owner."


In [6]:
df_a['manufacturer_age_interaction'] = df_a['manufacturer_freq'] * df_a['age']
df_a['one_owner_age_interaction'] = df_a['one_owner'].astype(int) * df_a['age']
df_a['one_owner_mileage_interaction'] = df_a['one_owner'].astype(int) * df_a['mileage']

df_a[['manufacturer_age_interaction', 'one_owner_age_interaction', 'one_owner_mileage_interaction']].describe()


,manufacturer_age_interaction,one_owner_age_interaction,one_owner_mileage_interaction
count,752900.000000,752900.000000,752900.000000
mean,0.414023,3.969142,24526.900042
std,0.411037,5.012954,34430.041403
min,0.013743,0.000000,0.000000
25%,0.183344,0.000000,0.000000
50%,0.317479,4.000000,9501.000000
75%,0.523635,6.000000,38617.250000
max,11.624703,111.000000,974302.000000


In [7]:
FEATURE_COLS_A = [
    'manufacturer', 'manufacturer_freq', 'model_freq', 'age', 'mileage',
    'transmission_simple', 'transmission_num_speeds', 'drivetrain', 'fuel_type',
    'accidents_or_damage', 'one_owner',
    'one_owner_age_interaction', 'one_owner_mileage_interaction',
    'manufacturer_age_interaction',
    'engine_displacement_l', 'engine_cylinders', 'engine_is_turbo', 'engine_is_hybrid',
    'city_mpg', 'highway_mpg', 'driver_reviews_num',
    'seller_rating_was_missing', 'driver_rating_was_missing',
    'price', 'price_log',
]

df_a_features = df_a[FEATURE_COLS_A].copy()
print("Final feature set shape:", df_a_features.shape)
print(df_a_features.dtypes)


Final feature set shape: (752900, 25)
manufacturer                      object
manufacturer_freq                float64
model_freq                       float64
age                                int64
mileage                          float64
transmission_simple               object
transmission_num_speeds          float64
drivetrain                        object
fuel_type                         object
accidents_or_damage                 bool
one_owner                           bool
one_owner_age_interaction          int64
one_owner_mileage_interaction    float64
manufacturer_age_interaction     float64
engine_displacement_l            float64
engine_cylinders                 float64
engine_is_turbo                    int64
engine_is_hybrid                   int64
city_mpg                         float64
highway_mpg                      float64
driver_reviews_num               float64
seller_rating_was_missing           bool
driver_rating_was_missing           bool
price              

In [8]:
out_path_a = "../data/processed/cars_features.csv"
df_a_features.to_csv(out_path_a, index=False)
print(f"Saved {df_a_features.shape[0]:,} rows x {df_a_features.shape[1]} columns to {out_path_a}")


Saved 752,900 rows x 25 columns to ../data/processed/cars_features.csv


## Part B — `used_cars` dataset (UK)

In [9]:
df_b = pd.read_csv("../data/processed/used_cars_clean.csv", low_memory=False)
print("Shape:", df_b.shape)


Shape: (97632, 10)


### Implementing the Part B summary table

| Feature | Call (from `03_insights.ipynb`) | What's done here |
|---|---|---|
| `Make` | KEEP | used as-is (9 categories), **plus** frequency-encoded so it can feed the interaction term below |
| `age` | KEEP, engineer | kept numeric, **plus** `age_squared` added -- Insight B2 found price variance narrows *non-linearly* with age ($12,976 → $3,575 std), which a plain linear term can't capture on its own |
| `mileage` | KEEP | used as-is (0.74 correlated with `age`, not redundant) |
| `tax` | **ENGINEER** | cast to category (48 discrete bands, not continuous) |
| `engineSize` | **ENGINEER** | cast to category (39 discrete values, standard displacements) |
| `transmission` | KEEP, caution | used as-is, **plus** an explicit interaction with `Make` (below) -- Insight B4 found the transmission price gap is "mostly (not entirely) a Make confound," the same "needs an interaction term" caution as `one_owner` in Part A |
| `fuelType` | **ENGINEER** | `Electric` + `Other` folded into a single `'Other'` bucket |
| `model` | **ENGINEER** | frequency-encoded, same approach as Part A |
| `price` | ENGINEER | `price_log` added as the modeling target |


In [10]:
df_b['age'] = CURRENT_YEAR - df_b['year']
df_b['age_squared'] = df_b['age'] ** 2
df_b['price_log'] = np.log1p(df_b['price'])

df_b['tax'] = df_b['tax'].astype('category')
df_b['engineSize'] = df_b['engineSize'].astype('category')

print(df_b[['age', 'age_squared', 'tax', 'engineSize']].dtypes)


age               int64
age_squared       int64
tax            category
engineSize     category
dtype: object


### `fuelType` — folding sparse categories (Insight B5's ENGINEER call)

`Electric` is 3 rows and `Other` is 239 rows out of 97,632 — neither has enough data to
support its own coefficient, so both fold into a single `'Other'` bucket.

In [11]:
print("Before:")
print(df_b['fuelType'].value_counts())

df_b['fuelType_grouped'] = df_b['fuelType'].replace({'Electric': 'Other'})

print("\nAfter:")
print(df_b['fuelType_grouped'].value_counts())


Before:
fuelType
Petrol      53978
Diesel      40407
Hybrid       3005
Other         239
Electric        3
Name: count, dtype: int64

After:
fuelType_grouped
Petrol    53978
Diesel    40407
Hybrid     3005
Other       242
Name: count, dtype: int64


### `model` and `Make` → frequency encoding (Insight B7's ENGINEER call)

Same reasoning and method as Part A: 194 unique values across only 97,632 rows (~500
rows/model on average, with a long tail well below that) is moderate but still worth
encoding rather than one-hotting directly. `Make` is frequency-encoded too, specifically to
feed the interaction term below -- with only 9 categories it doesn't need it otherwise.

In [12]:
model_freq_b = df_b['model'].value_counts(normalize=True)
df_b['model_freq'] = df_b['model'].map(model_freq_b)

make_freq_b = df_b['Make'].value_counts(normalize=True)
df_b['make_freq'] = df_b['Make'].map(make_freq_b)

print(df_b[['model', 'model_freq']].drop_duplicates().sort_values('model_freq', ascending=False).head(5))


         model  model_freq
58650   Fiesta    0.066658
716       Golf    0.049133
58651    Focus    0.046655
41041  C Class    0.037836
14892    Corsa    0.033647


### `transmission` × `Make` interaction (Insight B4's confound, actually built)

Automatic/semi-auto share rises steeply with brand prestige (59% Audi → 77% BMW → 89%
Mercedes-Benz), which is most of why "Automatic" looks like an expensive transmission type in
a flat, brand-blind comparison. This term gives a linear model a way to separate the
transmission effect from the brand effect, rather than conflating them.

In [13]:
df_b['is_auto_or_semi'] = df_b['transmission'].isin(['Automatic', 'Semi-Auto']).astype(int)
df_b['make_transmission_interaction'] = df_b['make_freq'] * df_b['is_auto_or_semi']

df_b[['is_auto_or_semi', 'make_transmission_interaction']].describe()


,is_auto_or_semi,make_transmission_interaction
count,97632.000000,97632.000000
mean,0.431488,0.051277
std,0.495286,0.062334
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.000000,0.108643
max,1.000000,0.182420


In [14]:
FEATURE_COLS_B = [
    'Make', 'make_freq', 'model_freq', 'age', 'age_squared', 'mileage',
    'tax', 'engineSize', 'transmission', 'is_auto_or_semi',
    'make_transmission_interaction', 'fuelType_grouped', 'mpg',
    'price', 'price_log',
]

df_b_features = df_b[FEATURE_COLS_B].copy()
print("Final feature set shape:", df_b_features.shape)
print(df_b_features.dtypes)


Final feature set shape: (97632, 15)
Make                               object
make_freq                         float64
model_freq                        float64
age                                 int64
age_squared                         int64
mileage                             int64
tax                              category
engineSize                       category
transmission                       object
is_auto_or_semi                     int64
make_transmission_interaction     float64
fuelType_grouped                   object
mpg                               float64
price                               int64
price_log                         float64
dtype: object


In [15]:
out_path_b = "../data/processed/used_cars_features.csv"
df_b_features.to_csv(out_path_b, index=False)
print(f"Saved {df_b_features.shape[0]:,} rows x {df_b_features.shape[1]} columns to {out_path_b}")


Saved 97,632 rows x 15 columns to ../data/processed/used_cars_features.csv


## Summary

| Dataset | Rows | Features kept | New in this version |
|---|---|---|---|
| `cars` | 752,900 | 18 base + 6 engineered (+ `price`, `price_log`) | Engine parsing (4 cols), `transmission_num_speeds`, `manufacturer_age_interaction`, `one_owner_age/mileage_interaction`, `seller_rating`/`driver_rating` `_was_missing` flags |
| `used_cars` | 97,632 | 9 base + 4 engineered (+ `price`, `price_log`) | `age_squared`, `make_freq`, `is_auto_or_semi`, `make_transmission_interaction` |

**Output files:**
- `../data/processed/cars_features.csv`
- `../data/processed/used_cars_features.csv`

**Not done here, on purpose:** no train/test split, no one-hot/target encoding of the
remaining categoricals, no scaling, no model fitting -- those choices belong in the modeling
notebook that consumes these two files, once it's clear which model (linear vs. tree-based)
goes first. Worth flagging for that notebook specifically: the frequency encodings here
(`model_freq`, `manufacturer_freq`, `make_freq`) are computed on the **full** dataset, not a
train split -- a soft form of leakage (no target information is used, only category counts,
so the risk is mild), but the modeling notebook should either accept that explicitly or
refit these frequency maps on its own train split before relying on them for a real
performance number.
